# 09 — Mechanistic Safety Audit: Refusal Direction in Qwen2.5-1.5B-Instruct

This notebook walks through the four-run abliteration pipeline:

1. **`refusal_direction`** — extract the refusal direction at layer 10
2. **`caa_steering`** — multi-layer sweep to identify the most effective layer
3. **`refusal_circuit_qwen`** — circuit patching to find causal heads
4. **`causal_scrubbing_refusal_qwen`** — faithfulness of the two-head circuit

**Prerequisites:** Run all four experiments first:
```bash
mech run --name refusal-direction-qwen
mech run --name caa-steering-qwen
mech run --name refusal-circuit-qwen
mech run --name causal-scrubbing-refusal-qwen
```
Then compile the audit report:
```bash
mech audit-refusal --refusal-run R --caa-run C --circuit-run P --scrub-run S
```

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

# Standard data/viz stack
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Project imports
from mech_interp.config import load_config
from mech_interp.storage import SQLiteResultStore
from mech_interp.analysis.refusal_audit import compile_refusal_audit

config = load_config()
store = SQLiteResultStore(config.project.database_path, config.project.artifact_dir)
print('Store initialised:', config.project.database_path)

In [ ]:
# ── Fill in your run IDs after running the four experiments ──
REFUSAL_RUN_ID = None   # e.g. 12
CAA_RUN_ID     = None   # e.g. 13
CIRCUIT_RUN_ID = None   # e.g. 14
SCRUB_RUN_ID   = None   # e.g. 15

# Quick check
for label, rid in [
    ('refusal', REFUSAL_RUN_ID),
    ('caa', CAA_RUN_ID),
    ('circuit', CIRCUIT_RUN_ID),
    ('scrub', SCRUB_RUN_ID),
]:
    if rid is None:
        print(f'WARNING: {label} run ID not set — fill in the cell above')
    else:
        r = store.get_result(rid)
        print(f'{label} run {rid}: status={r.status if r else "NOT FOUND"}')

## 1. Refusal Rate vs. Steering Coefficient (per layer)

In [ ]:
if CAA_RUN_ID is not None:
    caa_result = store.get_result(CAA_RUN_ID)
    assert caa_result is not None

    # Load layer_effectiveness artifact
    layer_eff_path = caa_result.artifacts.get('layer_effectiveness')
    if layer_eff_path and Path(layer_eff_path).is_file():
        layer_eff: dict = json.loads(Path(layer_eff_path).read_text())

        # Try to load steering results per layer for the sweep plot
        steering_results_path = caa_result.artifacts.get('steering_results')
        if steering_results_path and Path(steering_results_path).is_file():
            steering_data: dict = json.loads(Path(steering_results_path).read_text())

            fig, ax = plt.subplots(figsize=(9, 5))
            layers = sorted(int(k) for k in steering_data.keys() if k.isdigit())
            for layer in layers:
                layer_sweep = steering_data[str(layer)]
                coeffs = [r['coefficient'] for r in layer_sweep]
                rates  = [r['refusal_rate'] for r in layer_sweep]
                ax.plot(coeffs, rates, marker='o', label=f'Layer {layer}')

            ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
            ax.set_xlabel('Steering coefficient')
            ax.set_ylabel('Refusal rate')
            ax.set_title('Refusal Rate vs. Steering Coefficient — Qwen2.5-1.5B-Instruct')
            ax.legend()
            ax.set_ylim(-0.05, 1.05)
            plt.tight_layout()
            plt.show()
        else:
            print('steering_results artifact not found; run caa-steering-qwen first')
    else:
        print('layer_effectiveness artifact not found; run caa-steering-qwen first')
else:
    print('Set CAA_RUN_ID to plot this section')

## 2. Top-Head Importance Heatmap

In [ ]:
if CIRCUIT_RUN_ID is not None:
    circuit_result = store.get_result(CIRCUIT_RUN_ID)
    assert circuit_result is not None

    ranked_path = circuit_result.artifacts.get('patching_ranked_json')
    if ranked_path and Path(ranked_path).is_file():
        rows = json.loads(Path(ranked_path).read_text())

        df = pd.DataFrame(rows)
        print('Top 10 patching results:')
        print(df[['rank', 'hook_site', 'recovery_fraction', 'evidence_label']].head(10).to_string(index=False))

        # Simple bar chart of recovery fractions
        top = df.head(10)
        fig, ax = plt.subplots(figsize=(10, 4))
        colors = ['#2196F3' if e == 'causal evidence' else '#9E9E9E'
                  for e in top['evidence_label']]
        ax.barh(range(len(top)), top['recovery_fraction'], color=colors)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['hook_site'], fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel('Recovery fraction')
        ax.set_title('Circuit Patching — Top Hook Sites (blue = causal evidence)')
        plt.tight_layout()
        plt.show()
    else:
        print('patching_ranked_json artifact not found; run refusal-circuit-qwen first')
else:
    print('Set CIRCUIT_RUN_ID to plot this section')

## 3. Sample Generations Table

In [ ]:
if REFUSAL_RUN_ID is not None:
    refusal_result = store.get_result(REFUSAL_RUN_ID)
    assert refusal_result is not None

    intervention_path = refusal_result.artifacts.get('intervention_results')
    if intervention_path and Path(intervention_path).is_file():
        intervention_data: dict = json.loads(Path(intervention_path).read_text())

        rows_table = []
        for coeff_block in intervention_data.get('results', []):
            coeff = coeff_block['coefficient']
            for p in coeff_block.get('prompts', []):
                rows_table.append({
                    'coefficient': coeff,
                    'prompt': p['prompt'][:60] + '...' if len(p['prompt']) > 60 else p['prompt'],
                    'generation': p['generation'][:80] + '...' if len(p['generation']) > 80 else p['generation'],
                    'is_refusal': p['is_refusal'],
                })

        df_gen = pd.DataFrame(rows_table)
        # Show baseline (coeff=0) and most negative coefficient
        baseline = df_gen[df_gen['coefficient'] == 0.0]
        suppressed = df_gen[df_gen['coefficient'] == df_gen['coefficient'].min()]

        print('=== BASELINE (coeff=0) ===')
        print(baseline[['prompt', 'generation', 'is_refusal']].to_string(index=False))
        print()
        print(f'=== STEERING SUPPRESSED (coeff={suppressed["coefficient"].iloc[0]:+.1f}) ===')
        print(suppressed[['prompt', 'generation', 'is_refusal']].to_string(index=False))
    else:
        print('intervention_results artifact not found; run refusal-direction-qwen first')
else:
    print('Set REFUSAL_RUN_ID to show generation samples')

## 4. Compile Full Audit Report

In [ ]:
if all(x is not None for x in [REFUSAL_RUN_ID, CAA_RUN_ID, CIRCUIT_RUN_ID, SCRUB_RUN_ID]):
    report = compile_refusal_audit(
        refusal_run_id=REFUSAL_RUN_ID,
        caa_run_id=CAA_RUN_ID,
        circuit_run_id=CIRCUIT_RUN_ID,
        scrub_run_id=SCRUB_RUN_ID,
        store=store,
    )
    print(f'Model:               {report.model}')
    print(f'Best steering layer: {report.best_steering_layer}')
    print(f'Best coefficient:    {report.best_coefficient:+.1f}')
    print(f'Refusal rate shift:  {report.refusal_rate_shift_at_best:.3f}')
    print(f'Extraction quality:  {report.extraction_quality:.4f}')
    print(f'Circuit faithfulness:{report.circuit_faithfulness:.4f}')
    print(f'Top causal heads:    {report.top_causal_heads}')

    # Write outputs
    Path('refusal_audit.json').write_text(report.to_json())
    Path('refusal_audit.md').write_text(report.to_markdown())
    print('\nWrote refusal_audit.json + refusal_audit.md')

    # Display markdown
    from IPython.display import Markdown, display
    display(Markdown(report.to_markdown()))
else:
    print('Fill in all four run IDs above to compile the report')